In [ ]:
import os
import pandas as pd
import pickle
from prepare_sent_data import tokenize_sentence
import regex as re

def prepare_sentences(language): 
    """
    Takes the raw language corpus and converts each sentence into a list of words
    E.g. 
    Raw data: Jamais je n'épouserai cet hérétique, vous m'entendez?
    Output data: ['jamais', 'je', 'ne', 'épouserai', 'cet', 'hérétique', 'vous', 'me', 'entendez']
    
    Takes: language key 
    Writes the data to a pickle file 
    """
    # Load configuration
    config_path = "C:/Users/emill/Documents/GitHub/Coupe_Expansion/emillys_code/language_config.json"
    try:
        config_df = pd.read_json(config_path)
        config_df.set_index("Language", inplace=True)
        lang_cfg = config_df.loc[language]
        path = lang_cfg["Sentence Data"]
    except KeyError:
        print(f"Language '{language}' not found in configuration.")
        return
    except Exception as e:
        print(f"Failed to read config file: {e}")
        return

    # Prepare output directory
    output_dir = f"produced_data/{language}"
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"{language}_original_sentences.pkl")

    # Read and process sentences
    tokenized_sentences = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                sentence = line.strip()

                # Remove leading character if it's not a Unicode letter
                if sentence and not re.match(r'\p{L}', sentence[0]):
                    sentence = sentence[1:].lstrip()

                if not re.search(r'\p{L}', sentence, re.UNICODE):  # checks for any alphabetic Unicode letter
                    continue
                if any(char in sentence for char in ['(', ')', ':', '...']):
                    continue
                if sentence:
                    print(sentence)
                    tokenized = tokenize_sentence(sentence, language)
                    print(tokenized)
                    if tokenized:
                        tokenized_sentences.append(tokenized)
                
                # Optional: limit for testing
                if i >= 2000:
                    break
               
    except FileNotFoundError:
        print(f"Sentence data file not found: {path}")
        return

    # Save tokenized sentences
    try:
        with open(output_file, "wb") as f:
            pickle.dump(tokenized_sentences, f)
        print(f"✅ Saved tokenized sentences to '{output_file}'")
    except Exception as e:
        print(f"Failed to save tokenized data: {e}")


languages = ['FRA', 'DEU', 'JPN', 'CMN', 'VIE', 'YUE', 'ENG']

prepare_sentences(languages[0])


In [3]:
import csv
import pickle
import subprocess
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
import os
import unicodedata
from helpers import get_ipa_espeak, clean_ipa, load_config
import string
import re
import json


def count_word_freq(language):
    """
    Count word frequencies using espeak-ng IPA, save as JSON lookup dict.
    Allows lookup by word or by IPA.
    """
    # Set up paths
    base_dir = Path(f"produced_data/{language}")
    input_path = base_dir / f"{language}_original_sentences.pkl"
    lookup_path = base_dir / f"{language}_word_lookup.json"

    # Load espeak-ng language code
    espeak_code = load_config(language, key="IPA Code")


    # Load tokenized data
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")      
    
    with input_path.open("rb") as f:
        text = pickle.load(f)

    counter = Counter()
    lookup = defaultdict(list)

    for sentence in text:
        for word in sentence:
            # Convert to IPA clean
            word_ipa = get_ipa_espeak(word, espeak_code)
            # Clean IPA
            cleaned_ipa = clean_ipa(word_ipa, True, '', language)
            if cleaned_ipa:
                # Count the word's frequency in the corpus
                counter[(word, cleaned_ipa)] += 1

    # Build lookup
    for (word, cleaned_ipa), freq in counter.items():
        lookup[word].append({"ipa": cleaned_ipa, "freq": freq})
        lookup[cleaned_ipa].append({"word": word, "freq": freq})

    # Save lookup dict as JSON
    with lookup_path.open("w", encoding="utf-8") as f:
        json.dump(dict(lookup), f, ensure_ascii=False, indent=2)

    print(f"✅ Lookup dictionary for frquency counts saved to: {lookup_path}")


count_word_freq('FRA')


✅ Lookup dictionary for frquency counts saved to: produced_data\FRA\FRA_word_lookup.json


In [ ]:
import os
import subprocess
import pandas as pd
import pickle
import unicodedata
from helpers import load_config, clean_ipa, phoneme_tokenization
import regex as re
import json

def parse_to_pho_and_sylls(language):
    # Load tokenized text
    input_path = f"produced_data/{language}/{language}_original_sentences.pkl"
    if not os.path.exists(input_path):
        print(f"Input file '{input_path}' does not exist.")
        return
    with open(input_path, "rb") as f:
        text = pickle.load(f)

    phonemized_data = []  # list of lists
    # syllabized_data = []

    i = 0
    for sentence in text:
        sentence_phonemes = []
        # sentence_syllables = []

        for word in sentence:
            phonemes = phoneme_tokenization(word, language)
            if not phonemes: 
                continue

            #TODO tokenize to syllables

            sentence_phonemes.append(phonemes)
            # sentence_syllables.append(syllables)

        phonemized_data.append(sentence_phonemes)  # one list per sentence
        # syllabized_data.append(sentence_syllables)

    # Save data
    folder = f"produced_data/{language}"
    os.makedirs(folder, exist_ok=True)

    pho_output_path = f"{folder}/phonemized_{language}.json"
    sylls_output_path = f"{folder}/phonemized_{language}.json"

    with open(pho_output_path, 'w', encoding='utf-8') as f:
        json.dump(phonemized_data, f, ensure_ascii=False, indent=2)

    # with open(sylls_output_path, 'w', encoding='utf-8') as f:
        # json.dump(syllabized_data, f, ensure_ascii=False, indent=2)

    print(f"✅ Phonemization completed. Data saved to {pho_output_path}.")


parse_to_pho_and_sylls('FRA')

In [6]:
import pandas as pd
from helpers import load_config, get_ipa_espeak, clean_ipa
import re

def count_ling_units(language):
    """
    Takes a CSV with 'text' column, and appends:
    - IPA version
    - phoneme count
    """
    grapheme_pattern = re.compile(r"[^\s]")  # matches every non-space character (i.e. IPA "phoneme")
    
    # Load CSV and language config
    df = pd.read_csv("C:/Users/emill/Documents/GitHub/Coupe_Expansion/emillys_code/semantically_similar_texts/semantically_similar_texts.csv")
    espeak_code = load_config(language, "IPA Code")

    # Filter rows with respect to language
    df = df[df["language"] == language].copy()  

    # Result lists
    ipa_column = []
    n_phonemes_column = []


    for idx, row in df.iterrows():
        text = str(row["text"]).strip()

        # IPA transcription and phoneme an syllable count
        all_ipa = []
        all_phonemes = []

        for word in text.split():

            # Convert to ipa 
            word_ipa = get_ipa_espeak(word, espeak_code)

            # Clean IPA 
            cleaned_ipa = clean_ipa(word_ipa, True, '', language)

            if not cleaned_ipa:
                continue # removal of empty strings
            all_ipa.append(cleaned_ipa)

            # Tokenize into phonemes
            phonemes = [match.group() for match in grapheme_pattern.finditer(cleaned_ipa) if match.group() not in (' ', '')]
            all_phonemes.extend(phonemes)

        ipa_column.append(all_ipa)
        n_phonemes_column.append(len(all_phonemes))

    # Add to dataframe
    df["ipa"] = ipa_column
    df["n_phonemes"] = n_phonemes_column

    # Save result
    output_path = "semantically_similar_texts/ling_units_counts.csv"
    df.to_csv(output_path, sep="\t", index=False, encoding="utf-8")
    print(f"✅ Linguistic units counted and saved to: {output_path}")

count_ling_units('FRA')

✅ Linguistic units counted and saved to: semantically_similar_texts/ling_units_counts.csv
